# Module 2 — Modeling the World

*Live session, 1.5h.*

**Goal.** Looking at Chicago data through multiple lens, from
knowledge representation to operational analysis.

**The data.** Chicago's public procurement: 88,916 public contracts
and 1.2M business licences, real and uncleaned. We work on a sample — 46 contract rows,
54 licence rows — small enough to read line by line, and picked so that every
row is there for a reason.

**The tooling.** Everything runs inside this notebook. No server to start, no
database to install.

## Public Contracts

Chicago publishes every purchase it makes — construction, consultancy, dog and
cat food. The city's extract of 2026-08-11 — the one this sample is drawn from — has
**185,730 rows and 19 columns**, and the rows are not
contracts: each row is one **revision**. Those 185,730 revisions describe
**88,916 contracts**.

That distinction is the first modelling decision the data forces on us, and the
table below shows it before we have said a word about ontologies: contract 21085
is seven rows, and the columns that change between them are not the same columns
each time.

Our sample is every revision of contract 21085, the eight contracts whose dates
are hard to believe, and the contracts of two suppliers who also appear in the
licence data later.

Nothing is parsed. Every value is a string, the dates included — which is the
honest starting state, and the reason the implausible ones are still here to be
found.

In [ ]:
!pip install -q --disable-pip-version-check rdflib==7.6.0 pyshacl==0.40.1 owlrl==7.6.2 kuzu==0.11.3

import pandas as pd

from pathlib import Path

# This notebook is read in three places: its own folder in a checkout of the
# course repository, the same folder in the instructor repository — where the
# vocabularies sit two levels up — and Colab, where
# neither exists. pandas and rdflib both read a URL as readily as a path, so the
# only thing that has to change is the prefix.
RAW = ("https://raw.githubusercontent.com/giuseppefutia/kgtp-course/"
       "main/modules/02-modeling-the-world/")

def asset(name):
    """Return a readable location for a file that ships with this notebook."""
    for candidate in (Path(name), Path("../..") / name):
        if candidate.exists():
            return str(candidate)
    return RAW + name


pd.set_option("display.max_colwidth", None)
contracts = pd.read_csv(asset("data/contracts_sample.csv"), dtype=str).fillna("")

CONTRACT_ID = "Purchase Order (Contract) Number"

print(f"{len(contracts)} rows, {contracts[CONTRACT_ID].nunique()} contracts.")
print("Contract 21085 is seven of those rows. This is the granularity:\n")

one_contract = contracts[contracts[CONTRACT_ID] == "21085"]
one_contract.sort_values("Revision Number", key=lambda column: column.str.zfill(4))[
    ["Revision Number", "Contract Type", "Start Date", "End Date",
     "Approval Date", "Department", "Vendor Name", "Award Amount"]
].set_index("Revision Number")

### What each column carries

The table above shows what a *row* is. This one says what each **column** is
for, which a column name never does: `Revision Number` and `Approval Date` look
equally innocent until you know that one of them is why the other repeats.

In [ ]:
FIELDS = {
    "Purchase Order Description": "Free text. The only place the subject of the contract appears.",
    CONTRACT_ID: "Identifies the contract. Repeated once per revision, so it does not identify a row.",
    "Revision Number": "Which amendment this row is. Revisions belong to one contract; they are not separate contracts.",
    "Specification Number": "The city's internal specification code.",
    "Contract Type": "Belongs to the revision rather than to the contract: a contract can change type as it is amended.",
    "Start Date": "When the work begins. Text, like everything else — six of the eight implausible dates in this sample are here.",
    "End Date": "When the work ends. The other two are here: one contract running to the year 3030, and one to 2104.",
    "Approval Date": "When this revision was approved. The closest thing the file has to an award date.",
    "Department": "The contracting authority. It can change from one revision to the next, which is what contract 21085 shows.",
    "Vendor Name": "The supplier, as spelled on this row. Spelling varies between rows, and again between datasets.",
    "Vendor ID": "The city's supplier code. Where it is present it is stable, which is what makes a supplier traceable at all.",
    "Address 1": "Street address of the supplier.",
    "Address 2": "Second address line, usually empty.",
    "City": "City of the supplier.",
    "State": "State of the supplier.",
    "Zip": "Postal code of the supplier. Later in the module this becomes evidence: it is one of the signals used to decide that two spellings name one firm.",
    "Award Amount": "The value attached to this revision.",
    "Procurement Type": "How the supplier was chosen — competitive bid, sole source. The city's own vocabulary, not a shared one.",
    "Contract PDF": "Link to the signed document, where one exists.",
}

unknown = [c for c in contracts.columns if c not in FIELDS]
if unknown:
    raise KeyError(f"columns with no description: {unknown}")

# Revision 12 of contract 21085: the row where the department changes.
example = contracts[(contracts[CONTRACT_ID] == "21085")
                    & (contracts["Revision Number"] == "12")].iloc[0]

pd.DataFrame(
    [(c, FIELDS[c], example[c] or "—") for c in contracts.columns],
    columns=["column", "what it carries", "contract 21085, revision 12"],
).set_index("column")

## Business licences

Chicago licenses businesses to operate — food service, retail, raffles, pop-up
shops. The extract of 2026-08-12 has **1,203,437 rows and 37 columns**. We keep eleven.

In [ ]:
licences = pd.read_csv(asset("data/licenses_sample.csv"), dtype=str).fillna("")

FIELDS = {
    "ACCOUNT NUMBER": "Identifies the business, not the licence. One account carries many licences across many years, which is what makes a history possible.",
    "SITE NUMBER": "Which premises of that business. An account with three sites licenses each of them separately.",
    "LEGAL NAME": "The registered name. This is the string that has to meet the supplier name from the contracts file, and the two rarely agree exactly.",
    "DOING BUSINESS AS NAME": "The trading name, which can differ from the legal one in the very same row: LATITUDE INC. NFP trades as Latitude Chicago.",
    "ADDRESS": "Street address of the premises.",
    "ZIP CODE": "Postal code. Later it does real work — a shared postal code is one of the signals used to decide that two spellings name one firm.",
    "LICENSE NUMBER": "Identifies this licence.",
    "LICENSE DESCRIPTION": "What the licence permits.",
    "LICENSE STATUS": "Five codes, of which AAI covers 1,123,311 rows and AAC 78,652 (measured 2026-09-29). The file does not say what they mean; the codebook is published elsewhere.",
    "LICENSE TERM START DATE": "When the licence takes effect.",
    "LICENSE TERM EXPIRATION DATE": "When it lapses. With the start date it forms an interval — the thing a contract's own dates can be measured against, and the first property in this module that belongs to a relationship rather than to a thing.",
}

unknown = [c for c in FIELDS if c not in licences.columns]
if unknown:
    raise KeyError(f"described columns missing from the file: {unknown}")

example = licences[licences["LEGAL NAME"] == "RUNNING AWAY L.L.C."].iloc[0]

print(f"{len(licences)} rows, {licences['LEGAL NAME'].nunique()} businesses, "
      f"{licences['LICENSE DESCRIPTION'].nunique()} kinds of licence.\n")

pd.DataFrame(
    [(c, FIELDS[c], example[c] or "—") for c in FIELDS],
    columns=["column", "what it carries", "one licence of RUNNING AWAY L.L.C."],
).set_index("column")

## Public Contracts Ontology (somebody else's words)

The city publishes columns. **Public Contracts Ontology** is a vocabulary
published by people who have never heard of Chicago, used by procurement portals
across Europe. Mapping our columns onto it is not decoration: it is the only
reason a question asked elsewhere can reach our data.

It does not come alone. PCO declares that a contracting authority is a
`gr:BusinessEntity` — a class it does not define. That word belongs to
**GoodRelations**, so both vocabularies have to be present or the range of
`pc:contractingAuthority` points at nothing.

**A vocabulary is a lens as well.** PCO frames the administrative skeleton of a
contract — who awarded it, when, for how much, against which document — and that
frame is sharp. Outside it there is no word for a *revision*, none for a business
*licence*, and none for the fact that a supplier was the counterparty only
between two dates. Those are not oversights. They are what fell outside the
frame when this lens was ground, and we will have to decide what to do about
each one.

**So we extend it, instead of editing it.** A revision is not a different kind
of thing from a contract — it is the contract as it stood after one amendment —
so we declare `kgtp:ContractRecord` as a subclass of `pc:Contract`, in the same
language, in the same graph. Every term PCO offers a contract then applies to a
revision unchanged, and each revision carries exactly one award date, one
authority, one price. The multiplicity that follows will be ours, and declared.

The table below is **our mapping** — thirteen terms borrowed out of a hundred
and seventy-six, plus the three we just declared, each chosen for a column. The labels, kinds and constraints are read
from the files themselves, so they cannot drift from what the vocabularies actually say;
the rightmost column is the only part we wrote.

In [ ]:
from rdflib import Graph, RDF, RDFS, OWL, URIRef, BNode

vocab = Graph()
vocab.parse(asset("vocab/pc/public-contracts.ttl"), format="turtle")
vocab.parse(asset("vocab/gr/goodrelations.owl"), format="xml")

PC = "http://purl.org/procurement/public-contracts#"
GR = "http://purl.org/goodrelations/v1#"
KGTP = "http://kgtp.org/"

# What PCO has no word for, we have to declare ourselves — and declaring it in
# the same language, in the same graph, is the point: an ontology you borrow can
# be extended without being edited.
OUR_TERMS = """
@prefix kgtp: <http://kgtp.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix pc:   <http://purl.org/procurement/public-contracts#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

kgtp:ContractRecord a owl:Class ;
    rdfs:subClassOf pc:Contract ;
    rdfs:label "Contract revision"@en ;
    rdfs:comment "One row of the city's file: the contract as it stood after one amendment."@en .

kgtp:revisionOf a owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:label "revision of"@en ;
    rdfs:domain kgtp:ContractRecord ;
    rdfs:range pc:Contract .

kgtp:revisionNumber a owl:DatatypeProperty, owl:FunctionalProperty ;
    rdfs:label "revision number"@en ;
    rdfs:domain kgtp:ContractRecord ;
    rdfs:range xsd:string .
"""
vocab.parse(data=OUR_TERMS, format="turtle")

# The mapping, which is ours and not the vocabulary's.
MAPPING = {
    PC + "Contract":             "Purchase Order (Contract) Number (contract)",
    KGTP + "ContractRecord":     "Revision Number (contract)",
    KGTP + "revisionOf":         "Purchase Order (Contract) Number (contract)",
    KGTP + "revisionNumber":     "Revision Number (contract)",
    PC + "contractingAuthority": "Department (contract)",
    PC + "awardDate":            "Approval Date (contract)",
    PC + "startDate":            "Start Date (contract)",
    PC + "actualEndDate":        "End Date (contract)",
    PC + "Tender":               "Vendor Name (contract)",
    PC + "awardedTender":        "Vendor Name (contract)",
    PC + "bidder":               "Vendor ID (contract)",
    PC + "agreedPrice":          "Award Amount (contract)",
    PC + "agreement":            "Contract PDF (contract)",
    GR + "BusinessEntity":       "Vendor Name (contract), LEGAL NAME (licence)",
    GR + "legalName":            "Vendor Name (contract), LEGAL NAME (licence)",
    GR + "PriceSpecification":   "Award Amount (contract)",
}


absent = [t for t in MAPPING if (URIRef(t), None, None) not in vocab]
if absent:
    raise KeyError(f"terms not defined in any of the three vocabularies: {absent}")

PREFIXES = {PC: "pc:", GR: "gr:", KGTP: "kgtp:",
            "http://www.w3.org/2001/XMLSchema#": "xsd:",
            "http://www.w3.org/2000/01/rdf-schema#": "rdfs:",
            "http://xmlns.com/foaf/0.1/": "foaf:",
            "http://schema.org/": "schema:",
            "http://purl.org/dc/terms/": "dcterms:"}

def short(uri):
    text = str(uri)
    for long_form, prefix in PREFIXES.items():
        text = text.replace(long_form, prefix)
    return text

def label(term):
    for literal in vocab.objects(URIRef(term), RDFS.label):
        if getattr(literal, "language", None) == "en":
            return str(literal)
    return ""

def kind(term):
    types = set(vocab.objects(URIRef(term), RDF.type))
    if OWL.Class in types or RDFS.Class in types:
        return "class"
    base = "object property" if OWL.ObjectProperty in types else "data property"
    # Functional is the one that acts on the data rather than describing it.
    return base + (", functional" if OWL.FunctionalProperty in types else "")

def endpoints(term):
    """Render domain and range, unfolding the union that GoodRelations uses.

    A domain written as owl:unionOf is a blank node, so reading it naively
    prints nothing and the term looks unconstrained — which is the opposite of
    what it is.
    """
    def names(prop):
        out = []
        for value in vocab.objects(URIRef(term), prop):
            if isinstance(value, BNode):
                union = vocab.value(value, OWL.unionOf)
                out += [short(m) for m in vocab.items(union)] if union else []
            else:
                out.append(short(value))
        return " or ".join(out)

    domain, range_ = names(RDFS.domain), names(RDFS.range)
    if domain or range_:
        return f"{domain or '—'}  →  {range_ or '—'}"
    # No domain and no range of its own: the term is a narrowing of a broader
    # one and takes its constraints from there. Printing nothing would make it
    # look unconstrained, which is the opposite of what it is.
    parent = names(RDFS.subClassOf) or names(RDFS.subPropertyOf)
    return f"a kind of {parent}" if parent else ""

print(f"{len(vocab)} statements: two borrowed vocabularies and three terms of our own.\n")

pd.DataFrame([(short(t), label(t), kind(t), endpoints(t), MAPPING[t]) for t in MAPPING],
             columns=["term", "what it is called", "kind",
                      "domain → range", "our column"]).set_index("term")

## Representing Chicago Data in PCO Words

Now the mapping itself, by hand. This is where the decisions are — not in the
vocabulary, which only offers words, and not in the CSV, which only offers
values.

Three of those decisions shape everything that follows.

**Each row becomes a revision, and the values stay on it.** One row of the file
records the contract as it stood after one amendment: one approval date, one
start date, one department, one amount. Written onto the revision, every one of
them is single — which is what the file actually says. Written onto the contract
they would pile up, and a contract carrying seven award dates is not a fact about
Chicago: it is a fact about our shortcut.

**And the revisions do not even agree about when the work began.** Across the
2026-08-11 extract, 3,255 contracts of 69,522 carry more than one start date and
16,527 carry more than one end date — 4.7% and 23.5%. Contract 33757 is in this
sample and shows why that matters:

| revision | start date | approved |
|---|---|---|
| 0 | `03/01/0201` | 2016-07-29 |
| 4 | `03/01/2016` | 2017-07-11 |
| 5 | `03/01/2016` | 2017-09-15 |

The impossible date is on revision 0 alone. Two revisions later somebody typed
it correctly, and **the file carries both** — the error and its repair, each
stamped with when it was approved. Put the start date on the contract and you
have to choose one of them, which means throwing away either the fact or the
correction. Put it on the revision and you keep the sequence, and a question
like *what did we believe in 2016* still has an answer.

**The authority is lifted to the contract anyway, and that is deliberate.**
Nobody asks who approved amendment 9. The question is asked about the contract,
so the department has to be reachable from it — and a contract that changed hands
arrives carrying two. The award date is *not* lifted the same way: a contract is
awarded once, so the earliest of its revisions is the award and the rest are
amendments. One value, not seven. The difference between the two liftings is the
difference between a state that holds for a while and an event that happened.

**The department is identified by its spelling, because nothing else is
available.** Suppliers have a `Vendor ID`; departments have a name and nothing
more. So the name becomes the identity, and one body written two ways becomes two
entities. That is not a bug we are introducing — it is a limit of the source that
the model cannot repair, and it is why `DEPARTMENT OF CULTURAL AFFAIRS` and
`DEPARTMENT OF CULTURAL AFFAIRS AND SPECIAL EVENTS` sit in this graph as two
organisations.

The output prints the contract, then one of its seven revisions. Read them
together: the revision is where the file's values sit, one of each; the contract
is what we assembled, and the only place where a second authority appears.

In [ ]:
import re, datetime
from urllib.parse import quote
from rdflib import Graph, Literal, Namespace, URIRef, XSD, RDF
from rdflib.namespace import DCTERMS

KG = Namespace(KGTP)
PCO = Namespace(PC)
GRO = Namespace(GR)

US_DATE = re.compile(r"^(\d{1,2})/(\d{1,2})/(\d{1,4})$")

def as_date(value):
    """Turn the file's MM/DD/YYYY into a typed date, or return None.

    None means "say nothing": an empty cell becomes no triple at all, rather
    than a triple asserting emptiness. Nothing downstream can then mistake a
    value we never had for a value we checked.

    What this does *not* do is judge the date. 01/01/0204 converts happily into
    the year 204, because it is a well-formed date — absurd and well-formed are
    different questions, and this function only answers the second one.
    """
    match = US_DATE.match(value.strip())
    if not match:
        return None
    month, day, year = (int(part) for part in match.groups())
    try:
        return Literal(datetime.date(year, month, day), datatype=XSD.date)
    except ValueError:
        return None

slug = lambda text: re.sub(r"[^A-Za-z0-9]+", "-", text).strip("-").lower()

g = Graph()
for prefix, uri in [("pc", PCO), ("gr", GRO), ("kgtp", KG), ("dcterms", DCTERMS)]:
    g.bind(prefix, uri)

def business_entity(node, name):
    g.add((node, RDF.type, GRO.BusinessEntity))
    g.add((node, GRO.legalName, Literal(name)))

for contract_id, rows in contracts.groupby(CONTRACT_ID):
    contract = KG[f"contract-{contract_id}"]
    g.add((contract, RDF.type, PCO.Contract))
    g.add((contract, DCTERMS.identifier, Literal(contract_id)))
    g.add((contract, DCTERMS.description, Literal(rows.iloc[0]["Purchase Order Description"])))

    # One row of the file becomes one revision, and the dates go here — where
    # each of them is single, which is what the file actually says. Written onto
    # the contract instead they would pile up, and a contract with seven award
    # dates is not a fact about Chicago: it is a fact about our shortcut.
    for _, row in rows.iterrows():
        record = KG[f"record-{contract_id}-{row['Revision Number']}"]
        g.add((record, RDF.type, KG.ContractRecord))
        g.add((record, KG.revisionOf, contract))
        g.add((record, KG.revisionNumber, Literal(row["Revision Number"])))

        for column, prop in [("Approval Date", PCO.awardDate),
                             ("Start Date", PCO.startDate),
                             ("End Date", PCO.actualEndDate)]:
            typed = as_date(row[column])
            if typed is not None:
                g.add((record, prop, typed))

        if row["Department"]:
            g.add((record, PCO.contractingAuthority, KG[f"dept-{slug(row['Department'])}"]))

        if row["Award Amount"]:
            price = KG[f"amount-{contract_id}-{row['Revision Number']}"]
            g.add((record, PCO.agreedPrice, price))
            g.add((price, RDF.type, GRO.PriceSpecification))
            g.add((price, GRO.hasCurrencyValue,
                   Literal(float(row["Award Amount"]), datatype=XSD.float)))
            g.add((price, GRO.hasCurrency, Literal("USD")))

    # And now the decision this module turns on. A question is never asked about
    # a revision — nobody asks "who approved amendment 9". It is asked about the
    # contract. So the authority is lifted from the revisions to the contract,
    # and a contract that changed hands arrives carrying two of them.
    #
    # The award date is not lifted the same way: a contract is awarded once, so
    # the earliest of its revisions is the award and the rest are amendments.
    # One value, not seven. The difference between the two liftings is the
    # difference between a state that holds for a while and an event.
    for name in sorted({row["Department"] for _, row in rows.iterrows() if row["Department"]}):
        department = KG[f"dept-{slug(name)}"]
        g.add((contract, PCO.contractingAuthority, department))
        # The department has no identifier anywhere in this file — only a
        # spelling. So the spelling has to become the identity, and one body
        # written two ways becomes two entities. Nothing here can undo that.
        business_entity(department, name)

    awarded = sorted(d for d in (as_date(v) for v in rows["Approval Date"]) if d is not None)
    if awarded:
        g.add((contract, PCO.awardDate, awarded[0]))

    # The supplier hangs off the contract through a tender: pc:bidder has
    # rdfs:domain pc:Tender, so attaching it to the contract directly would not
    # raise an error — it would quietly retype the contract as a tender.
    tender = KG[f"tender-{contract_id}"]
    g.add((contract, PCO.awardedTender, tender))
    g.add((tender, RDF.type, PCO.Tender))
    for vendor_id in sorted(set(rows["Vendor ID"])):
        if not vendor_id:
            continue
        supplier = KG[f"org-{vendor_id}"]
        g.add((tender, PCO.bidder, supplier))
        for spelling in sorted(set(rows.loc[rows["Vendor ID"] == vendor_id, "Vendor Name"])):
            business_entity(supplier, spelling)

    for pdf in sorted(set(rows["Contract PDF"])):
        if not pdf.startswith("http"):
            continue
        # The city's own document links carry braces, which a URI cannot hold:
        # rdflib accepts one and then fails when asked to write it out. Escaping
        # them is the smallest example in this notebook of a value that does not
        # enter the model until somebody decides what to do with it.
        g.add((contract, PCO.agreement, URIRef(quote(pdf, safe=":/?=&"))))

print(f"{len(g)} statements: "
      f"{len(set(g.subjects(RDF.type, PCO.Contract)))} contracts, "
      f"{len(set(g.subjects(RDF.type, KG.ContractRecord)))} revisions, "
      f"{len(set(g.subjects(RDF.type, GRO.BusinessEntity)))} business entities.\n")

def around(subject, depth=1, stop=()):
    """Collect a subject's statements, and those of the nodes it points at.

    `stop` is the reason this takes an argument at all. kgtp:revisionOf points
    from a revision back to its contract, so following links blindly prints the
    contract a second time — which reads as a duplicate in the output and is
    really just the graph being a graph.
    """
    out = Graph()
    for prefix, uri in g.namespaces():
        out.bind(prefix, uri)
    frontier, seen = {subject}, set(stop)
    for _ in range(depth + 1):
        following = set()
        for node in frontier:
            if node in seen:
                continue
            seen.add(node)
            for triple in g.triples((node, None, None)):
                out.add(triple)
                if isinstance(triple[2], URIRef) and str(triple[2]).startswith(str(KG)):
                    following.add(triple[2])
        frontier = following
    return out

# The contract, then one of its revisions. Read them together: the revision is
# where the file's values sit, one of each; the contract is what we assembled,
# and the only place a second authority appears.
print("The contract:\n")
print(around(KG["contract-21085"], depth=0).serialize(format="turtle"))
print("One of its seven revisions:\n")
print(around(KG["record-21085-12"], stop=[KG["contract-21085"]])
      .serialize(format="turtle"))

## The first lens: OWL
### What follows from what you wrote

The mapping is done and nothing has judged it. Hand the graph to a reasoner,
together with the vocabularies it was written against, and it works out what
else must be true — then writes that down next to your data.

**What it acts on is one line of PCO:**

```turtle
pc:contractingAuthority a owl:FunctionalProperty .
```

That word needs unpacking, because everything below follows from it. Calling a
property *functional* is a statement about the property itself, made once in the
vocabulary and binding on every use of it: **for any one subject, this property
leads to at most one value.** It is not a rule to be enforced. It is a fact
being asserted — the way "a person has one date of birth" is a fact, not a
policy. PCO makes twenty-six such assertions.

Now put the two together. The vocabulary asserts that a contract leads to at
most one authority. Our graph gives contract 21085 two of them. The reasoner
cannot call the data wrong, and cannot call the vocabulary wrong, so it takes
the only exit left:

> **"At most one" does not mean *do not give me two*. It means *if you see two,
> they are one*.**

Watch that happen on a contract for dog and cat food which moved from the Police
to Animal Care — a transfer that makes perfect sense in the world, and that this
lens cannot see as a transfer at all.

Three things in the output are worth more than the merge itself.

**What it gave us, which is real.** 721 statements of ours, 2,334 of vocabulary,
3,055 in; 10,206 out. Of the 7,151 added, **351 are about our own nodes and say
something we did not write** — and most of them are useful. 135 of our dates are
now reachable as `dcterms:date`, 46 prices as `pc:contractPrice`, 23 suppliers
as `pc:tender`, 18 documents as `pc:attachment`. That is the reasoner earning
its keep: we wrote the precise term, and anyone who knows only the general one
can now find our data. It is the whole argument for mapping onto somebody else's
vocabulary, carried out automatically.

**And one of those 7,151 is a disaster.** Not a different mechanism — the same
one. `rdfs:subPropertyOf` generalised our dates; `owl:FunctionalProperty` merged
two departments. Both are PCO describing itself, both fire without being asked.

**Which is the finding.** Not that reasoning is dangerous: that *the output does
not distinguish the gift from the damage*. 351 inferences about our data, 350 of
them welcome, one of them false, and nothing in the result — no flag, no
ordering, no report — tells you which is which. You either knew what to look
for, or you did not look.

**The silence.** No error, no warning. PCO gives the reasoner nothing it could
object to: eleven distinct OWL terms, every one of them descriptive — no
cardinality, no `owl:differentFrom`, not a single `owl:disjointWith`. A
vocabulary meant to be adopted by many publishers cannot afford to refuse
anybody's data, and refusing is somebody else's job.

So the blind spot is not that the lens got it wrong. It is that **from where
this lens stands, nothing went wrong** — there is no report, because a
conclusion is not a finding.

In [ ]:
import time
from collections import Counter
from owlrl import DeductiveClosure, OWLRL_Semantics
from rdflib import OWL

# The reasoner rewrites the graph it is handed, in place. Everything below
# therefore happens on a copy: the data graph has to survive this cell intact,
# because the shapes are run against it afterwards and have to see what we wrote
# rather than what was concluded from it.
reasoned = Graph()
for triple in g:
    reasoned.add(triple)
for triple in vocab:          # the axioms are what the reasoner acts on
    reasoned.add(triple)

before = set(reasoned)
started = time.time()
DeductiveClosure(OWLRL_Semantics).expand(reasoned)
added = set(reasoned) - before
print(f"{len(before)} statements in, {len(reasoned)} out — "
      f"{len(added)} added in {time.time() - started:.1f}s.\n")

# Most of the closure is bookkeeping — every resource is an owl:Thing, every
# term is the same as itself — and it cannot be switched off: it is the OWL RL
# semantics, not an option. What can be done, and what anybody working with a
# reasoner actually does, is look only at the part that touches your own names.
ours = [(s, p, o) for s, p, o in added
        if str(s).startswith(KGTP) and p != RDF.type and p != OWL.sameAs]
abbrev = lambda n: (str(n).replace(KGTP, "kgtp:").replace(PC, "pc:").replace(GR, "gr:")
                    .replace("http://purl.org/dc/terms/", "dcterms:")
                    .replace("http://www.w3.org/2001/XMLSchema#", "xsd:"))

print(f"Of those, {len(ours)} say something new about our own nodes:\n")
for predicate, count in Counter(p for _, p, _ in ours).most_common(6):
    example = next((s, o) for s, p, o in ours if p == predicate)
    print(f"   {count:>4}  {abbrev(predicate)}")
    print(f"         e.g.  {abbrev(example[0])}  {abbrev(predicate)}  {abbrev(example[1])}")

# The gift, made concrete. We wrote pc:awardDate; PCO declares it a kind of
# dcterms:date; so a query that has never heard of public procurement finds it.
GENERAL = """PREFIX dcterms: <http://purl.org/dc/terms/>
             SELECT ?when WHERE { ?contract dcterms:date ?when }"""
print("\n'Give me any date you hold', asked by someone who does not know PCO:")
for label, graph in [("before", g), ("after ", reasoned)]:
    print(f"   {label}: {len(list(graph.query(GENERAL)))} answers")

# And the damage, from the same closure, by the same mechanism.
merges = {tuple(sorted((str(s), str(o))))
          for s, o in reasoned.subject_objects(OWL.sameAs)
          if s != o and str(s).startswith(KGTP) and str(o).startswith(KGTP)}
print(f"\nIdentities the reasoner asserted on its own: {len(merges)}\n")
for a, b in sorted(merges):
    print(f"   {a.replace(KGTP, 'kgtp:')}")
    print(f"   {b.replace(KGTP, 'kgtp:')}")

POLICE = KG["dept-chicago-police-department"]
ASK = """PREFIX gr: <http://purl.org/goodrelations/v1#>
         SELECT ?name WHERE { ?dept gr:legalName ?name }"""
print("\nWhat is the police department called?\n")
for label, graph in [("before", g), ("after ", reasoned)]:
    names = [str(r.name) for r in graph.query(ASK, initBindings={"dept": POLICE})]
    print(f"   {label}: {names}")

## The second lens: SHACL
### A rule you declare, and a report you read

Nothing so far has an opinion about the year 3030. The vocabulary does not: PCO
says an end date is an `xsd:date`, and `3030-11-30` is one. The mapping did not
either — it converted the string and moved on, because *absurd* and *well-formed*
are different questions and only the second one has an answer in the file. And
the reasoner, a moment ago, walked straight past it.

So the opinion has to be written down somewhere, deliberately, as its own
artifact. That is what SHACL is for.

The cell starts by printing one revision out of the graph, so that what follows
is visibly about statements you can see rather than about a report. Then two
shapes, and **they are here for opposite reasons**.

The first fires. Start and end dates are typed by a person reading a contract
and reach legitimately into 2074 for long housing covenants, so the bound is not
"the range of the dataset" but "a year somebody could have meant" — a weaker
claim, deliberately.

The second finds nothing, and that is its job. An approval date is stamped by
the city when a human approves a revision; measured over all 185,730 rows on
2026-09-30, every one falls between 1993-01-01 and 2026-08-07. **A shape that
passes is a result**: the dates the city stamps itself are clean, and the ones a
person typed are not.

**One detail in the shapes repays reading.** Both name two target classes,
`pc:Contract` and `kgtp:ContractRecord`. We declared the second a subclass of
the first — but that axiom lives in the vocabulary, not in the data graph, and
SHACL follows subclasses only where the data says so. Aim at `pc:Contract`
alone and this cell reports **`conforms`**, because every date is on a revision
and it never looked at one.

That is the failure mode to carry out of the room. A shape reports on the nodes
it reached. It has nothing to say about the ones it did not, and it says that in
the same word it uses for success.

In [ ]:
from pyshacl import validate

# Before running anything: the statements a shape is about to object to. This is
# the graph itself, not a report about it — the year 3030 sits in there as a
# perfectly well-formed xsd:date, and nothing up to now has had an opinion.
print("One revision, as it stands in the graph:\n")
print(around(KG["record-106913-2"], depth=0).serialize(format="turtle"))

SHAPES = """
@prefix sh:   <http://www.w3.org/ns/shacl#> .
@prefix pc:   <http://purl.org/procurement/public-contracts#> .
@prefix kgtp: <http://kgtp.org/> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
@prefix kg:   <http://kgtp.org/shapes#> .

# Both classes are named, and that is not belt and braces. kgtp:ContractRecord
# is declared a subclass of pc:Contract, but that axiom lives in the vocabulary
# and not in the data graph, and SHACL follows subclasses only where the data
# says so. Target pc:Contract alone and this whole cell reports "conforms" —
# because the dates are on the revisions, and it never looked at one.
kg:ContractPeriodIsPlausible a sh:NodeShape ;
    sh:targetClass pc:Contract, kgtp:ContractRecord ;
    sh:property [
        sh:path pc:startDate ;
        sh:minInclusive "1900-01-01"^^xsd:date ;
        sh:maxInclusive "2100-12-31"^^xsd:date ;
        sh:message "Start date is not a year anyone could have meant." ;
    ] ,
    [
        sh:path pc:actualEndDate ;
        sh:minInclusive "1900-01-01"^^xsd:date ;
        sh:maxInclusive "2100-12-31"^^xsd:date ;
        sh:message "End date is not a year anyone could have meant." ;
    ] .

# And one that passes. An approval date is stamped by the city when a human
# approves a revision, so it cannot precede the dataset and cannot be in the
# future: measured over all 185,730 rows on 2026-09-30, every one of them falls
# between 1993-01-01 and 2026-08-07, with no exceptions at all.
#
# It is here precisely because it finds nothing. "Conforms" from a shape that
# was aimed at the right nodes is a result — the dates the city stamps itself
# are clean, and the ones a person typed are not.
kg:ApprovalIsPlausible a sh:NodeShape ;
    sh:targetClass pc:Contract, kgtp:ContractRecord ;
    sh:property [
        sh:path pc:awardDate ;
        sh:minInclusive "1993-01-01"^^xsd:date ;
        sh:maxInclusive "2026-12-31"^^xsd:date ;
        sh:message "Approval date falls outside the period this dataset covers." ;
    ] .
"""

def check(shapes_turtle, data=None):
    """Validate the data graph against one shapes graph and return the report.

    The shapes stay in a graph of their own. Parsing them into the data would
    also work — pyshacl finds them either way — and it would teach that a
    constraint is just more data about the same subject, which is the first
    thing to unlearn here. What we want on the screen is two graphs meeting,
    one of which can be replaced without touching the other.
    """
    shapes = Graph()
    shapes.parse(data=shapes_turtle, format="turtle")
    conforms, report, _ = validate(data if data is not None else g,
                                   shacl_graph=shapes, inference="none", advanced=True)
    return conforms, report

SH = Namespace("http://www.w3.org/ns/shacl#")

conforms, report = check(SHAPES)
print(f"conforms: {conforms}\n")

# The message is half of what a shape is for. A report that gives you a node and
# a property tells you where to look; the message is the only place the author
# can say what they meant, and it is what distinguishes a violation you can act
# on from a line of machine output.
for result in sorted(report.subjects(RDF.type, SH.ValidationResult), key=str):
    node = str(report.value(result, SH.focusNode)).replace(KGTP, "kgtp:")
    path = str(report.value(result, SH.resultPath)).split("#")[-1]
    print(f"   {node:<24} {path:<14} {report.value(result, SH.value)}")
    print(f"   {'':<24} {report.value(result, SH.resultMessage)}\n")

## The same sentence, said to the other lens

`pc:contractingAuthority a owl:FunctionalProperty` and `sh:maxCount 1` on
`pc:contractingAuthority` are the same sentence in two formalisms — same
subject, same number, same claim. Run both on the same graph and they do not
disagree at the margin. They disagree about whether anything is wrong at all.

|  | OWL concluded | SHACL reports |
|---|---|---|
| two authorities on contract 21085 | they are the same organisation | **1 node** breaks the rule, and names it |

One merges in silence, one reports and names the node. Neither is a better
version of the other: they are answers to different questions, and the question
was never "is this data valid" — it was *what do you want to happen if this turns
out to be false.*

**And read the second half of the output, which is the more useful one.**
`sh:maxCount 1` on `pc:awardDate` finds nothing. Every node carries at most one
award date — because we put the dates on the revisions, where the file puts them,
and lifted only the earliest to the contract.

Had we flattened the revisions onto the contract instead, that shape would have
reported contract 21085 with **seven** award dates. The data would be identical.
The violation would have been entirely of our own making, and the right repair
would not have been to the data but to the model.

So a report has two possible readings, and SHACL does not tell you which one you
are holding:

- **the data is wrong** — go and look at the record
- **the model is wrong** — the rule is right and the graph is shaped badly

The first is what people expect from validation. The second is what it is
actually most useful for, and the only way to tell them apart is to know what you
meant when you built the graph.

In [ ]:
CARDINALITY = """
@prefix sh:   <http://www.w3.org/ns/shacl#> .
@prefix pc:   <http://purl.org/procurement/public-contracts#> .
@prefix kgtp: <http://kgtp.org/> .
@prefix kg:   <http://kgtp.org/shapes#> .

# The same sentence PCO already declares as owl:FunctionalProperty, said again in
# the other formalism. No new information: what changes is what the machine is
# being asked to do with it.
kg:OneAuthority a sh:NodeShape ;
    sh:targetClass pc:Contract, kgtp:ContractRecord ;
    sh:property [
        sh:path pc:contractingAuthority ;
        sh:maxCount 1 ;
        sh:message "This node names more than one contracting authority." ;
    ] .

# And the same sentence about the award date, which is where the modelling
# decision gets its report card.
kg:OneAwardDate a sh:NodeShape ;
    sh:targetClass pc:Contract, kgtp:ContractRecord ;
    sh:property [
        sh:path pc:awardDate ;
        sh:maxCount 1 ;
        sh:message "This node carries more than one award date." ;
    ] .
"""

conforms, report = check(CARDINALITY)
print(f"conforms: {conforms}\n")

contracts_total = len(set(g.subjects(RDF.type, PCO.Contract)))
records_total = len(set(g.subjects(RDF.type, KG.ContractRecord)))
reported = {}
for result in report.subjects(RDF.type, SH.ValidationResult):
    node = report.value(result, SH.focusNode)
    path = report.value(result, SH.resultPath)
    reported.setdefault(str(path).split("#")[-1], []).append(
        (str(node).replace(KGTP, "kgtp:"), len(list(g.objects(node, path))),
         str(report.value(result, SH.resultMessage))))

for path in ["contractingAuthority", "awardDate"]:
    hits = sorted(reported.get(path, []))
    print(f"{path}: {len(hits)} of {contracts_total} contracts and "
          f"{records_total} revisions")
    for node, count, message in hits:
        print(f"   {node:<24} {count} values")
        print(f"   {'':<24} {message}")
    if not hits:
        print("   nothing — every node carries at most one")
    print()

### Where Core stops

Every shape so far walks one step from a node and checks what it finds. Here is a
rule that cannot be written that way:

> a contract's revisions should be numbered 0, 1, 2 … with nothing missing

Read what it needs. **Count** the revisions. Take the **highest** number any of
them carries. **Compare** the two. Core SHACL has no counting beyond
`sh:minCount` and `sh:maxCount`, no arithmetic, and no way to hold one value
while it works out another. The constraint is not awkward to express — it is
outside the language.

`sh:sparql` is the door out. Read the shape from the outside in:

| | |
|---|---|
| `sh:targetClass pc:Contract` | run this on every contract, one at a time |
| `sh:sparql [ … ]` | the constraint is a query rather than one of SHACL's built-ins |
| `SELECT $this …` | **`$this` is the contract currently under test.** SHACL binds it before the query runs, so the query never has to find it |
| `?revision kgtp:revisionOf $this` | reach the revisions of *that* contract — the one hop Core could also do |
| `FILTER (regex(?text, "^[0-9]+$"))` | keep only revisions numbered with digits. Not defensive habit: 17,636 rows of the 2026-08-11 extract are numbered `D404171` or `P00739`, and a sequence check has nothing to say about those |
| `COUNT(?revision)`, `MAX(?number)` | how many there are, and the highest number any of them carries — the two things Core cannot compute |
| `GROUP BY $this` | one answer per contract, not per revision |
| `HAVING (COUNT(…) != MAX(…) + 1)` | the rule itself. Seven revisions numbered 0…6 is consecutive; seven whose highest is 12 is not |

**And then the inversion that catches everybody.** A query normally returns what
you asked for. Here, **a returned row is a violation** — the SELECT describes
the broken case, not the healthy one, and a contract that is fine produces no
row at all. Writing it the other way round, which is the natural instinct,
reports every correct contract in the dataset.

The aggregates it computes are available to the message: `{?present}` and
`{?highest}` in `sh:message` are filled in per violation, which no Core
constraint can do.

**And look at what it finds on the contract we have been following all session.**
Contract 21085 has seven revisions and its highest is numbered 12. The table in
the very first cell showed 0, 7, 8, 9, 10, 11, 12 — and six numbers were missing
in plain sight before anybody had said the word *ontology*.

It is not rare: **7,878 of the 88,916 contracts** in the full file have a gap,
8.9% of them, measured 2026-09-30.

**Now the part worth arguing about.** Is this a violation at all? The city
publishes the current state of each revision that still stands; superseded ones
are simply not in the file. On that reading nothing is broken — the file is a
**snapshot**, and we were reading it as a **history**.

So a report has a third possible meaning, on top of the two from the last cell:

- the data is wrong
- the model is wrong
- **your assumption about what the file is was wrong**

The third is the one nobody writes a shape for, because you only discover the
assumption by tripping over it. This shape is how you trip.

In [ ]:
# Every shape so far walks one step from a node and checks what it finds there.
# This rule cannot be written that way:
#
#     a contract's revisions should be numbered 0, 1, 2 … with nothing missing
#
# Read what that needs. Count the revisions. Take the highest number any of them
# carries. Compare the two. Core SHACL has no counting beyond sh:minCount and
# sh:maxCount, no arithmetic, and no way to hold one value while it computes
# another — so the constraint is not merely awkward to express, it is outside the
# language. sh:sparql is the door out: a SELECT whose result rows *are* the
# violations, with $this bound to the node being checked.
REVISION_SEQUENCE = """
@prefix sh:   <http://www.w3.org/ns/shacl#> .
@prefix pc:   <http://purl.org/procurement/public-contracts#> .
@prefix kgtp: <http://kgtp.org/> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
@prefix kg:   <http://kgtp.org/shapes#> .

kg:RevisionsAreConsecutive a sh:NodeShape ;
    sh:targetClass pc:Contract ;
    sh:sparql [
        a sh:SPARQLConstraint ;
        sh:message "Revisions are not consecutive: {?present} present, highest numbered {?highest}." ;
        sh:select \"\"\"
            SELECT $this (COUNT(?revision) AS ?present) (MAX(?number) AS ?highest)
            WHERE {
                ?revision <http://kgtp.org/revisionOf> $this ;
                          <http://kgtp.org/revisionNumber> ?text .
                # Why this guard is here, and the measurement behind it,
                # are in the prose above. Repeating them would be a second
                # copy to keep true.
                FILTER (regex(?text, "^[0-9]+$"))
                BIND (xsd:integer(?text) AS ?number)
            }
            GROUP BY $this
            HAVING (COUNT(?revision) != MAX(?number) + 1)
        \"\"\" ;
    ] .
"""

conforms, report = check(REVISION_SEQUENCE)
print(f"conforms: {conforms}\n")
for result in sorted(report.subjects(RDF.type, SH.ValidationResult), key=str):
    node = str(report.value(result, SH.focusNode)).replace(KGTP, "kgtp:")
    print(f"   {node:<26} {report.value(result, SH.resultMessage)}")

## The third lens: the property graph

The first two lenses argued about contract 21085 because neither had anywhere to
put the one thing that explains it: **the two departments are not simultaneous.**
The Police held it from 2010, Animal Care from 2016. Say that, and there is
nothing left to disagree about.

So the question for this lens is not *what does it forbid* — it is **where does
a fact like "between these dates" live**.

The answer is the edge, and that is the whole claim. A relationship in a
property graph is a thing in its own right: it has a type, a direction, and
properties of its own. Not a pair of endpoints with a label, but somewhere you
can write on.

**A note on the store, so it is not mistaken for an argument.** Kuzu wants a
schema declared before it will accept a row, and the cell below shows it
refusing three writes. That is worth seeing once — a check that happens before
the value reaches anybody is a different kind of check from the two we have run
— but it is **a property of this product, not of the paradigm**: Neo4j takes
whatever you give it, and every relational database has had this since 1970.
Nothing in what follows depends on it.

Note also what the schema does *not* stop. `3030-11-30` goes in without a
murmur: it is a well-formed date, and a schema checks shape, not sense. Every
one of the eight implausible dates would enter here untouched.

In [ ]:
import kuzu, os, tempfile

def fresh_database():
    """Open an empty store in a throwaway directory.

    Rebuilt on every run, so nothing in this notebook depends on state a
    previous run left behind.
    """
    return kuzu.Connection(kuzu.Database(os.path.join(tempfile.mkdtemp(), "graph")))

SCHEMA = [
    "CREATE NODE TABLE Contract(id STRING, description STRING, PRIMARY KEY(id))",
    "CREATE NODE TABLE Department(name STRING, PRIMARY KEY(name))",
    "CREATE NODE TABLE Supplier(id STRING, name STRING, PRIMARY KEY(id))",
    # The interval belongs on the edge. Put it on the contract and a contract
    # can only ever have one department; put it on the department and it makes
    # no sense at all. It belongs to the fact that joins them, which is the
    # thing that was true for a while and then stopped being true.
    "CREATE REL TABLE ASSIGNED_TO(FROM Department TO Contract, validFrom DATE, validTo DATE)",
    "CREATE REL TABLE SUPPLIED_BY(FROM Contract TO Supplier, validFrom DATE, validTo DATE)",
]

# The refusals are shown on a store of their own, so the probe rows never reach
# the one we actually load.
scratch = fresh_database()
for statement in SCHEMA:
    scratch.execute(statement)

def attempt(description, statement):
    try:
        scratch.execute(statement)
        print(f"   accepted   {description}")
    except Exception as error:
        print(f"   REFUSED    {description}")
        print(f"              {str(error).splitlines()[0][:92]}")

print("What this particular store refuses, before anything is loaded:\n")
attempt("a contract with the columns the schema declares",
        "CREATE (:Contract {id: 'probe', description: 'fine'})")
attempt("a property nobody declared",
        "CREATE (:Contract {id: 'probe-2', colour: 'red'})")
attempt("the string '01/01/0204' in a DATE column",
        """MATCH (c:Contract {id: 'probe'})
           CREATE (:Department {name: 'probe-dept'})-[:ASSIGNED_TO {validFrom: '01/01/0204'}]->(c)""")
attempt("an edge whose endpoints are not the declared ones",
        """MATCH (a:Contract {id: 'probe'}), (b:Contract {id: 'probe'})
           CREATE (a)-[:ASSIGNED_TO]->(b)""")
attempt("the year 3030, which is a perfectly good date",
        """MATCH (c:Contract {id: 'probe'})
           CREATE (:Department {name: 'probe-dept-2'})-[:ASSIGNED_TO {validFrom: date('3030-11-30')}]->(c)""")

## The same fact, a third time

Contract 21085 has two departments. The first lens made that into an identity;
the second made it into a violation. Here it is neither. It is a sequence, and
the cell below draws it.

Nothing was repaired to get there. The same seven revisions, read the same way.
What changed is that there is now **somewhere to put the time** — and that
somewhere is the edge.

### Three ways to say "the Police held this contract from 2010 to 2016"

    property graph
        (contract 21085) ──ASSIGNED_TO {validFrom, validTo}──▶ (police)
                              the dates sit on the arrow

    RDF, reified
        (contract 21085) ──▶ (assignment-7f3a) ──▶ (police)
                                  validFrom 2010-07-19
                                  validTo   2016-06-06
                              a node standing for the relationship — which is
                              not a thing anybody in Chicago would recognise

    RDF-star
        << kgtp:contract-21085  pc:contractingAuthority  kgtp:dept-police >>
               kgtp:validFrom "2010-07-19" ;
               kgtp:validTo   "2016-06-06" .
                              the statement itself becomes the subject

**RDF-star is the real answer, and it should be said plainly** — RDF noticed
this gap and closed it: a triple can be the subject of other triples, with no
invented node. Two caveats keep it from ending the argument here.

The first is practical: the library this notebook runs on does not parse it.
`rdflib 7.6.0` rejects `<<` and registers no `turtle-star` parser (checked
2026-10-01). The standard is ahead of the tooling you will actually meet.

The second is structural, and it is the more interesting one. RDF-star annotates
a **statement**. If a relationship holds, stops, and holds again — a supplier
that leaves and comes back — there is still only one triple, and two periods
cannot be hung on it separately without going back to reification. A property
graph simply has two edges. Here the two departments are two different triples,
so RDF-star would cope; the day a department returns, it would not.

### Why the edge, and not either end

Try to move `validFrom` in the schema. On the contract, it forces one department
for the whole life of the contract — the assumption that started this argument.
On the department, it is nonsense: a department does not have a validity. The
dates belong to neither end. They belong to the fact that joins them, which was
true for six years and then stopped.

**And the counterweight, so the lens gets no more credit than it earned.** This
store took `3030-11-30` without a murmur. It did not solve the date problem. It
solved a different one, and the two are not interchangeable.

In [ ]:
import datetime

kz = fresh_database()
for statement in SCHEMA:
    kz.execute(statement)

def to_date(value):
    """Return a datetime.date, or None when the cell holds no usable date."""
    literal = as_date(value)                # the same conversion the RDF used
    return literal.toPython() if literal is not None else None

def periods(rows, column):
    """Work out when each value held, from the dates on the revisions.

    A revision records that a department was in charge when it was approved; it
    never records when that stopped. So the end of one period is read off the
    start of the next, and the last period is left open — which is a claim about
    the present, and the only one the file supports.

    Returns nothing for a contract whose revisions carry no approval date at
    all: an interval invented for such a contract would be indistinguishable
    from one that was measured.
    """
    first_seen = {}
    for _, row in rows.iterrows():
        value, approved = row[column], to_date(row["Approval Date"])
        if not value or approved is None:
            continue
        first_seen[value] = min(first_seen.get(value, approved), approved)

    ordered = sorted(first_seen.items(), key=lambda pair: pair[1])
    return [(value, start, ordered[i + 1][1] if i + 1 < len(ordered) else None)
            for i, (value, start) in enumerate(ordered)]

for po, rows in contracts.groupby(CONTRACT_ID):
    kz.execute("CREATE (:Contract {id: $id, description: $text})",
               {"id": po, "text": rows.iloc[0]["Purchase Order Description"]})

    for name, start, end in periods(rows, "Department"):
        kz.execute("MERGE (:Department {name: $name})", {"name": name})
        kz.execute("""MATCH (d:Department {name: $name}), (c:Contract {id: $id})
                      CREATE (d)-[:ASSIGNED_TO {validFrom: $since, validTo: $until}]->(c)""",
                   {"name": name, "id": po, "since": start, "until": end})

    for vendor_id, start, end in periods(rows, "Vendor ID"):
        spelling = rows.loc[rows["Vendor ID"] == vendor_id, "Vendor Name"].iloc[0]
        kz.execute("MERGE (:Supplier {id: $id, name: $name})",
                   {"id": vendor_id, "name": spelling})
        kz.execute("""MATCH (c:Contract {id: $po}), (s:Supplier {id: $id})
                      CREATE (c)-[:SUPPLIED_BY {validFrom: $since, validTo: $until}]->(s)""",
                   {"po": po, "id": vendor_id, "since": start, "until": end})

counts = kz.execute("""MATCH (c:Contract) RETURN count(c)""").get_next()[0]
edges = kz.execute("""MATCH ()-[r:ASSIGNED_TO]->() RETURN count(r)""").get_next()[0]
print(f"{counts} contracts, {edges} department assignments.\n")

result = kz.execute("""
    MATCH (d:Department)-[r:ASSIGNED_TO]->(c:Contract {id: '21085'})
    RETURN d.name, r.validFrom, r.validTo
    ORDER BY r.validFrom
""")
spells = []
while result.has_next():
    name, start, end = result.get_next()
    spells.append((name, start, end))

def timeline(spells, width=58):
    """Draw the spells as a bar, so the shape of the history is visible.

    Drawn rather than listed because two rows of dates read as two facts, and a
    bar reads as one fact with a hinge in it — which is what it is.
    """
    first = min(s for _, s, _ in spells)
    last = max(e or datetime.date.today() for _, _, e in spells)
    span = (last - first).days or 1
    at = lambda d: int((d - first).days / span * (width - 1))
    bar = [" "] * width
    for _, start, _ in spells:
        bar[at(start)] = "|"
    print(f"   {first}{'':<{width - 22}}{last}")
    print("   " + "".join("-" if ch == " " else ch for ch in bar))
    for name, start, end in spells:
        left = at(start)
        print(f"   {'':<{left}}^ {start} → {end or 'still in charge'}  {name}")

print("Contract 21085, the same fact the first two lenses argued about:\n")
timeline(spells)

## Edges that carry evidence

The interval on `ASSIGNED_TO` said something about Chicago: a department really
was in charge between those dates. The properties below say nothing about
Chicago at all. They say what **we** did, and why.

    score 0.93   ['NORM_ADDRESS', 'SAME_ZIP']
       LATTITUDE INC NFP      [contracts]
       LATITUDE INC. NFP      [licences]

**The three signals, since the output names them and nothing has said what they
are.** A name alone never decides: it has to be corroborated, and the weaker the
corroboration the better the name has to be.

| signal | what it checks | name must reach |
|---|---|---|
| `SAME_ADDRESS` | the two sides give the **same address string** — strong evidence, two businesses rarely share a doorway by accident | 0.30 |
| `NORM_ADDRESS` | the same address **once the unit is stripped** — `11 E ADAMS ST STE 1500` and `11 E ADAMS ST 1500` become one. Weaker: two firms can share a building | 0.50 |
| `SAME_ZIP` | only the **postcode** matches — weak, thousands of businesses share one, so almost the whole burden falls on the name | 0.80 |

The thresholds are not calibrated against anything. They are somebody's
judgement, written down where it can be read and changed — which is the most
that can be said for them, and more than the first lens offered.

That is the same claim the first lens made about the Police and Animal Care —
*these two are one organisation* — with one difference. This one arrives as a
**proposal carrying its reasons**: a name score, and the two independent signals
that corroborated it. You can disagree with it. You can find it, because it is
an edge; you can weigh it, because it has a number; you can remove it, because
nothing else was rewritten when it was added.

The merge OWL performed left no such trace. There was no edge to inspect, no
score to doubt and nothing to undo, because a conclusion is not a record of a
decision — it is just more data, indistinguishable from what you loaded.

**Two kinds of property on an edge, then, and they earn their place differently.**
`validFrom` is about the world and has nowhere else it could live. `score` and
`methods` are about our own process, and they are what make a judgement
reviewable instead of merely made.

### The one that did not fire, which is the more useful half

`ROSELAND CHRISTIAN HEALTH M 01` and `ROSELAND CHRISTIAN MINISTRIES CENTER` are
the same organisation. This rule does not merge them, and it is not a bug: the
names score **0.58**, the only corroboration available is a shared postal code,
and a postal code is weak evidence — thousands of businesses share one — so the
rule demands a name above 0.8 before it will accept it. It declines. Correctly,
on the evidence it has.

In the full dataset they *are* merged — but never to each other. Measured on the
whole graph, 2026-10-01:

    the pair this sample asks about

        ROSELAND CHRISTIAN HEALTH M 01  ──╳──  ROSELAND CHRISTIAN MINISTRIES CENTER
                   [contracts]          0.58           [licences]
                                    no edge: a postcode demands 0.80

    the chain the full dataset actually has — four spellings, group L695

        ROSELAND CHRISTIAN HEALTH M 01            [contracts]
               │ 0.82  SAME_ZIP
        ROSELAND CHRISTIAN HEALTH MINISTRIE   ←┐
               │ 0.88                          │  the two spellings
        ROSELAND CHRISTIAN MINISTRIES          │  our sample does not have
               │ 0.91                         ←┘
        ROSELAND CHRISTIAN MINISTRIES CENTER      [licences]

Two consequences worth carrying out of the room.

**Resolution is not pairwise.** The identity that joins the contract side to the
licence side is built from three links, and *the pair you care about is not one
of them*. A cluster can be right while no single edge in it looks convincing,
and no edge in it is the one you would have thought to check.

**And the sample changed the answer.** We kept two spellings out of four, and a
correct merge disappeared — not because the rule got worse, but because the
stepping stones were gone.

In [ ]:
import re
from itertools import combinations

# Two tables more: the parties as they are spelled in each file, and the edge
# that says we think two spellings are one organisation. The edge carries the
# reasons — which is the whole point of adding it rather than merging the nodes.
kz.execute("CREATE NODE TABLE Party(name STRING, source STRING, PRIMARY KEY(name))")
kz.execute("""CREATE REL TABLE RESOLVED_WITH(FROM Party TO Party,
                                             score DOUBLE, methods STRING[])""")

LEGAL_FORM = re.compile(r"\b(?:co|ltd|inc|corp|llc|llp|nfp|company|incorporated)\b", re.I)
UNIT = re.compile(r"\b(?:ste|suite|apt|unit|fl|floor|#)\b.*$|\s+\d+\s+\d+$|\s+[A-Z]?\d{2,4}$", re.I)

def name_similarity(left, right):
    """Score two names between 0 and 1, ignoring their legal form.

    Sørensen–Dice over character bigrams, which is what the production pipeline
    uses. Stripping "INC", "LLC" and the rest first matters more than the metric
    does: without it, two unrelated companies share the suffix and start out
    looking alike.
    """
    def bigrams(name):
        cleaned = re.sub(r"[^a-z0-9]+", "", LEGAL_FORM.sub(" ", name.lower()))
        return {cleaned[i:i + 2] for i in range(len(cleaned) - 1)}
    a, b = bigrams(left), bigrams(right)
    return 2 * len(a & b) / (len(a) + len(b)) if a and b else 0.0

without_unit = lambda address: re.sub(r"\s+", " ", UNIT.sub("", address.upper())).strip()

parties = {}
for _, row in contracts.drop_duplicates("Vendor ID").iterrows():
    parties[row["Vendor Name"]] = ("contracts", {row["Address 1"].upper()}, {row["Zip"]})
for name, rows in licences.groupby("LEGAL NAME"):
    parties[name] = ("licences", set(rows["ADDRESS"].str.upper()), set(rows["ZIP CODE"]))

for name, (source, _, _) in parties.items():
    kz.execute("CREATE (:Party {name: $name, source: $source})",
               {"name": name, "source": source})

# The name alone never decides. It has to be corroborated, and the weaker the
# corroboration the higher the name has to score: a shared address is strong
# evidence, a shared postal code is weak — thousands of businesses share one.
RULES = [("SAME_ADDRESS", 0.3, lambda a, b: bool(a & b)),
         ("NORM_ADDRESS", 0.5, lambda a, b: bool({without_unit(x) for x in a} &
                                                 {without_unit(x) for x in b})),
         ("SAME_ZIP",     0.8, None)]

for left, right in combinations(parties, 2):
    (_, addresses_l, zips_l), (_, addresses_r, zips_r) = parties[left], parties[right]
    score = name_similarity(left, right)
    methods = []
    for label, floor, matches in RULES:
        shares = matches(addresses_l, addresses_r) if matches else bool(zips_l & zips_r)
        if shares and score >= floor:
            methods.append(label)
    if methods:
        kz.execute("""MATCH (a:Party {name: $left}), (b:Party {name: $right})
                      CREATE (a)-[:RESOLVED_WITH {score: $score, methods: $methods}]->(b)""",
                   {"left": left, "right": right, "score": score, "methods": methods})

# The candidates, before anything is decided about them. Fifteen spellings from
# two files, and the whole exercise is which of them name one organisation.
print(f"{len(parties)} spellings in play:\n")
for name, (source, _, postcodes) in sorted(parties.items(), key=lambda kv: (kv[1][0], kv[0])):
    print(f"   [{source:<9}] {name:<42} {', '.join(sorted(postcodes))}")

print(f"\n{kz.execute('MATCH ()-[r:RESOLVED_WITH]->() RETURN count(r)').get_next()[0]} merges proposed.\n")

result = kz.execute("""
    MATCH (a:Party)-[r:RESOLVED_WITH]->(b:Party)
    RETURN a.name, a.source, b.name, b.source, r.score, r.methods
    ORDER BY r.score DESC
""")
while result.has_next():
    left, source_l, right, source_r, score, methods = result.get_next()
    print(f"   score {score:.2f}   {methods}")
    print(f"      {left}   [{source_l}]")
    print(f"      {right}   [{source_r}]\n")

## The question neither file can answer

Everything so far has been one file at a time. Here is a question that is not in
either of them:

> **Was this supplier licensed when the city signed with it?**

The contract side knows when things were signed. The licence side knows when
licences ran. Neither knows they are about the same organisation — that is the
edge we computed in the last cell, and it is the only reason the question can be
asked at all. Three pieces, and the answer does not exist until they are in one
graph.

**A different slice, because a different question.** This cell loads a second
pair of files: one supplier, 564 contract rows and 8 licence rows. The 46-row
sample is sized to be read line by line; this one is sized to be computed on,
and pretending one sample fits both would be a worse lie than loading two.

**And note what the licence file turns out to be.** Its eight rows carry one
`LICENSE NUMBER`, repeated — one licence, eight annual terms. That is exactly
the shape the contracts file has, hidden exactly the same way, and we only see
it because we were bitten by it there first. So the node is the *term*, as the
node was the *revision*.

**Read the identifiers before the merges.** Each file names this body with a
code of its own, and the codes are the first thing the output prints:

| file | identifier | value |
|---|---|---|
| contracts | `Vendor ID` | `63437745V` — **and** `90455112P` |
| licences | `ACCOUNT NUMBER` | `211320` |

Two supplier codes for one body on the contract side, one account number on the
licence side, and **no code appears on both**. The two namespaces were minted by
different parts of the city and have nothing to do with each other. That is the
whole reason any of this has to be done on names and addresses: there is no key
to join on, and there never was one.

The join comes out three times, and the third one is a free lesson: the city's
own file carries the spelling `SGA YOUTH & FAMILY SERVICES, NFP|CLEANED-UP` —
somebody's cleanup marker, published. 8,851 rows across 164 names carry it
(measured 2026-09-30). Our rules reunite it with the plain spelling at 0.82.

### What the walk finds

    last licence expired         2015-04-15
    approvals signed after that  477
    the first of them            2015-04-30, fifteen days later
    the last                     2026-07-10

Eight consecutive annual renewals, no gaps, every one of them issued. Then
nothing — and 477 contract approvals over the following eleven years.

**Now be careful about what has been shown, because this is the whole skill.**

What the graph supports: *this body held a licence without interruption from
2007 to 2015, did not renew, and has signed 477 contract approvals since.* Every
step of that is a statement you can walk back to a row.

What it does not support: *it operated unlicensed*. The file records one licence
type. It does not say whether a city contractor needs one, nor whether this body
moved to a registration that is not in this dataset.

So the output is **a question with a name and two dates attached** — not a
verdict. That is the same lesson as the end date in 2104: the graph finds the
place where your judgement is needed, and does not supply it.

### And this is what the edges bought

Walk the query backwards and every step carries its own evidence. The approval
is a date on a revision. The licence term is an interval on its own node. The
identity joining them is an edge with `score: 1.00` and the methods that
produced it.

Ask the same thing of the first lens and you get an answer with nothing behind
it: when the reasoner decided the Police and an animal shelter were one
organisation, it left no edge, no score and no row to return to. Here, if the
answer is wrong, **the reason is reachable** — and on a merge that scored 0.38
rather than 1.00, it would be the first place to look.

In [ ]:
integration_contracts = pd.read_csv(asset("data/integration_contracts.csv"), dtype=str).fillna("")
integration_licences = pd.read_csv(asset("data/integration_licenses.csv"), dtype=str).fillna("")

print(f"{len(integration_contracts)} contract rows, {len(integration_licences)} licences, "
      f"one supplier.\n")

ink = fresh_database()
for statement in [
    # A party is kept per source, not per name. The two files may spell a body
    # identically and still be two records of it, and deciding they are one is
    # the thing this cell is about: it has to be an edge we can point at, not an
    # assumption built into the primary key.
    "CREATE NODE TABLE Party(id STRING, name STRING, source STRING, PRIMARY KEY(id))",
    "CREATE NODE TABLE Contract(id STRING, description STRING, PRIMARY KEY(id))",
    "CREATE NODE TABLE Revision(id STRING, approvedOn DATE, PRIMARY KEY(id))",
    # One licence, many terms -- exactly the shape the contracts file has, and
    # hidden the same way: LICENSE NUMBER is repeated on all eight rows, as the
    # contract number is repeated on every revision. The node is the term.
    "CREATE NODE TABLE LicenceTerm(id STRING, number STRING, validFrom DATE, validUntil DATE, PRIMARY KEY(id))",
    "CREATE REL TABLE AMENDS(FROM Revision TO Contract)",
    "CREATE REL TABLE AWARDED_TO(FROM Contract TO Party)",
    "CREATE REL TABLE HOLDS(FROM Party TO LicenceTerm)",
    "CREATE REL TABLE RESOLVED_WITH(FROM Party TO Party, score DOUBLE, methods STRING[])",
]:
    ink.execute(statement)

sides = {}

def party(source, name, address, postcode, identifier):
    """Record a spelling, with whatever each file uses to identify it.

    The identifiers are carried because of what they turn out to say. The
    contracts file identifies a supplier by `Vendor ID`, the licence file by
    `ACCOUNT NUMBER`, and the two namespaces have nothing to do with each
    other — which is the reason any of this has to be done on names at all.
    """
    key = f"{source}:{name}"
    if key not in sides:
        sides[key] = (source, name, {address.upper()}, {postcode}, {identifier})
        ink.execute("CREATE (:Party {id: $id, name: $name, source: $source})",
                    {"id": key, "name": name, "source": source})
    else:
        sides[key][2].add(address.upper())
        sides[key][3].add(postcode)
        sides[key][4].add(identifier)
    return key

for _, row in integration_contracts.iterrows():
    contract_id = row[CONTRACT_ID]
    # MERGE matches on every property written in the pattern, so the key goes in
    # the pattern and the rest in ON CREATE SET — otherwise the second revision
    # of a contract tries to create the contract again.
    ink.execute("""MERGE (c:Contract {id: $id})
                   ON CREATE SET c.description = $text""",
                {"id": contract_id, "text": row["Purchase Order Description"]})
    approved = to_date(row["Approval Date"])
    if approved is not None:
        revision = f"{contract_id}-{row['Revision Number']}"
        ink.execute("""MERGE (r:Revision {id: $id})
                       ON CREATE SET r.approvedOn = $approved""",
                    {"id": revision, "approved": approved})
        ink.execute("""MATCH (r:Revision {id: $rev}), (c:Contract {id: $id})
                       MERGE (r)-[:AMENDS]->(c)""", {"rev": revision, "id": contract_id})
    key = party("contracts", row["Vendor Name"], row["Address 1"], row["Zip"],
                row["Vendor ID"])
    ink.execute("""MATCH (c:Contract {id: $id}), (p:Party {id: $party})
                   MERGE (c)-[:AWARDED_TO]->(p)""", {"id": contract_id, "party": key})

for _, row in integration_licences.iterrows():
    key = party("licences", row["LEGAL NAME"], row["ADDRESS"], row["ZIP CODE"],
                row["ACCOUNT NUMBER"])
    start, until = to_date(row["LICENSE TERM START DATE"]), to_date(row["LICENSE TERM EXPIRATION DATE"])
    if start is None or until is None:
        continue
    ink.execute("""MERGE (l:LicenceTerm {id: $id})
                   ON CREATE SET l.number = $number, l.validFrom = $since, l.validUntil = $until""",
                {"id": row["ID"], "number": row["LICENSE NUMBER"], "since": start, "until": until})
    ink.execute("""MATCH (p:Party {id: $party}), (l:LicenceTerm {id: $id})
                   MERGE (p)-[:HOLDS]->(l)""", {"party": key, "id": row["ID"]})

# The same rules as before, on this slice. Nothing here knows that the two files
# describe one organisation until these edges say so.
for left, right in combinations(sides, 2):
    (_, name_l, addresses_l, zips_l, _) = sides[left]
    (_, name_r, addresses_r, zips_r, _) = sides[right]
    score = name_similarity(name_l, name_r)
    methods = []
    for label, floor, matches in RULES:
        shares = matches(addresses_l, addresses_r) if matches else bool(zips_l & zips_r)
        if shares and score >= floor:
            methods.append(label)
    if methods:
        ink.execute("""MATCH (a:Party {id: $left}), (b:Party {id: $right})
                       CREATE (a)-[:RESOLVED_WITH {score: $score, methods: $methods}]->(b)""",
                    {"left": left, "right": right, "score": score, "methods": methods})

# Each spelling with the identifier its own file gives it. Read the three
# columns before reading the merges: the contracts file knows this body by two
# different supplier codes, the licence file by an account number, and no code
# appears on both sides. There is nothing to join on but the words.
print("The spellings, and what each file calls them:\n")
for key, (source, name, _, postcodes, identifiers) in sorted(sides.items()):
    label = "Vendor ID" if source == "contracts" else "Account"
    print(f"   [{source:<9}] {name}")
    print(f"   {'':<12} {label}: {', '.join(sorted(identifiers))}   "
          f"postcode: {', '.join(sorted(postcodes))}")

print("\nThe join, and what it rests on:\n")
joined = ink.execute("""
    MATCH (a:Party)-[r:RESOLVED_WITH]->(b:Party)
    RETURN a.name, a.source, b.name, b.source, r.score, r.methods
""")
while joined.has_next():
    name_l, source_l, name_r, source_r, score, methods = joined.get_next()
    print(f"   score {score:.2f}  {methods}")
    print(f"      {name_l}   [{source_l}]")
    print(f"      {name_r}   [{source_r}]\n")

print("Walking it — revision, contract, supplier, resolved identity, licence terms:\n")
answer = ink.execute("""
    MATCH (rev:Revision)-[:AMENDS]->(c:Contract)-[:AWARDED_TO]->(p:Party)
    MATCH (p)-[:RESOLVED_WITH]-(q:Party)-[:HOLDS]->(l:LicenceTerm)
    WITH rev, max(l.validUntil) AS lastLicence
    WHERE rev.approvedOn > lastLicence
    RETURN count(rev), min(rev.approvedOn), max(rev.approvedOn), max(lastLicence)
""")
signatures, first, last, expiry = answer.get_next()
print(f"   last licence expired            {expiry}")
print(f"   approvals signed after that     {signatures}")
print(f"   the first of them               {first}")
print(f"   the last                        {last}")

## What the three lenses did, and what they cannot do

One fact — a contract for dog and cat food that changed department — and here is
everything the session did with it, plus the other cases it dragged in:

|  | `"01/01/0204"` as text | the year 3030 | two departments on one contract | one firm, two spellings |
|---|---|---|---|---|
| **the store's schema**, when a row is written | refuses it | lets it in | takes it: two edges | — |
| **shapes**, when you choose to run them | — | names the node | names the node | — |
| **the reasoner**, when you ask a question | — | silent | declares them one organisation | silent |
| **the property graph**, once edges can carry things | — | — | a history with dates | a proposal with a score |

No column is covered by one row. No row covers every column. That is the whole
finding, and it is not a compromise between three tools — it is what each of
them is *for*.

**A rule of thumb, which is worth more than the syntax.** Ask yourself, before
writing any rule: *if this turned out to be false in my data, what do I want to
happen?*

- *I want to be told, with the name of the offender* → **shapes**
- *I want it never to get in* → **a typed schema**, where the store offers one
- *I want the system to conclude something* → **the reasoner**
- *It cannot be false; it is true by definition* → **the reasoner**

Or, shorter: **the reasoner for data that leaves the building, shapes for data
arriving at the door, a typed schema for data being written down.**

### Two things this session did not do

**It fixed nothing.** Every violation found today is still in the graph, on
purpose. Finding and repairing are
different jobs, and conflating them is how a repair gets made before anybody has
understood what was broken.

**And it did not make the graph correct.** Five rules were written, three of them
fired, and all five were written by somebody who already suspected what they
would find. The eight implausible dates
were found because a conversion looked odd and a person went looking; no shape
would have been written for the year 3030 before somebody had seen one.

> **Shapes state what you already know. They do not discover it.**

Discovering it is a different problem, and the harder one.

### And the blind spot, one last time

Each lens has one, and from inside a lens the blind spot looks like empty space.
The reasoner did not know it had merged the Police with an animal shelter — it
had no means of noticing, and no report in which the noticing could have
appeared.

So the defence is never a better lens. It is a second one, pointed at the same
fact, that fails differently.